# Módulo 8 — Del modelo al servicio

Los siete módulos anteriores entrenan y evalúan dieciséis detectores. Ninguno deja algo con lo que puntuar una transacción nueva.

Es un hueco que contradice todo el hilo de los Módulos 5 a 7: se calcularon umbrales, se midió cuánta plata salva cada punto de operación y se discutió recalibración, sin que existiera un objeto capaz de recibir una transacción y responder.

Este módulo construye ese objeto y mide qué cuesta usarlo.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import RobustScaler

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.operations.temporal import get_temporal_data
from src.serving.explain import global_importance, occlusion_attribution
from src.serving.package import DetectorPackage, build_package
from src.serving.predict import alert_rate, score_transactions, top_alerts
from src.serving.run_serving import carga_diaria, medir_latencia
from src.unsupervised.families import GMMDensity

## 1. Qué tiene que viajar junto

Un modelo serializado no alcanza. Para decidir sobre una transacción hacen falta cinco cosas, y si alguna viaja por separado el sistema se rompe en silencio:

| Componente | Por qué no puede faltar |
|---|---|
| **Detector ajustado** | El modelo propiamente dicho |
| **Escalador** | Ajustado *solo* con entrenamiento. Reajustarlo sobre el tráfico del día cambiaría la escala bajo los pies del detector |
| **Scores de calibración** | Convierten un score crudo en p-valor conforme. Sin ellos el score es un número sin unidades |
| **Nivel alpha** | La tasa de falsas alarmas que el sistema promete |
| **Contrato de columnas** | El nombre y orden de las features |

El último es el más traicionero. Un `DataFrame` con las mismas columnas en otro orden produce scores perfectamente plausibles y completamente equivocados, **sin lanzar ninguna excepción**.

In [ ]:
data = get_temporal_data()
X_train, X_calib = data["X_train"], data["X_calib"]
X_test, y_test, steps = data["X_test"], data["y_test"], data["steps_test"]

scaler = RobustScaler().fit(X_train)
detector = GMMDensity().fit(scaler.transform(X_train))

package = build_package(
    detector, scaler, scaler.transform(X_calib), list(X_train.columns),
    alpha=0.01, detector_name="gmm_density",
    extra={"cutoff_step": int(data["cutoff_step"]), "n_train": int(len(X_train))},
)
package.metadata

### Por qué se empaqueta Gaussian Mixture y no el mejor por PR-AUC

Sobre el split temporal, Deep SVDD gana el ranking (0.368 contra 0.263). Pero el Módulo 5 mostró que su umbral promete 0,1% de falsas alarmas y entrega **35%**: es indesplegable tal cual está.

GMM queda segundo en ranking y primero en lo que importa para operar — calibración (razón 1,4 contra 354) y dinero salvado (944 M contra 833 M). **Elegir el modelo por su puesto en el ranking y dar el umbral por sentado es exactamente el error que el Módulo 5 hace visible.**

## 2. El contrato de columnas

La validación no es defensiva por costumbre: es la única forma de que un error de integración falle ruidosamente en vez de devolver números creíbles.

In [ ]:
normal = X_test.iloc[[0]]
revuelto = normal[list(reversed(normal.columns))]

print("mismo resultado con las columnas al revés:",
      np.allclose(package.score(normal), package.score(revuelto)))

for descripcion, entrada in [
    ("falta una columna", normal.drop(columns=[normal.columns[0]])),
    ("sobra una columna", normal.assign(columna_nueva=1.0)),
]:
    try:
        package.score(entrada)
        print(f"{descripcion}: NO se detectó (mal)")
    except ValueError as e:
        print(f"{descripcion}: rechazado -> {str(e)[:60]}")

## 3. Score crudo contra p-valor

El score crudo solo sirve para ordenar transacciones entre sí. El p-valor conforme tiene unidades interpretables: es la fracción del tráfico legítimo de calibración que resulta al menos tan anómala.

Un score de 14,3 no significa nada por sí solo. Un p-valor de 0,0004 dice que menos de una transacción legítima de cada dos mil se ve así de rara.

In [ ]:
muestra = X_test.iloc[:8]
comparacion = pd.DataFrame({
    "score": package.score(muestra),
    "p_valor": package.p_values(muestra),
    "alerta": package.predict(muestra),
})
print(f"umbral (alpha={package.alpha}): {package.threshold:.4f}")
comparacion

## 4. Persistencia: lo que se guarda es lo que se carga

Una verificación de ida y vuelta que conviene tener siempre. Si el paquete recargado no reproduce los scores originales, algo del estado no se serializó — y es el tipo de error que aparece recién en producción.

In [ ]:
ruta = package.save()
recargado = DetectorPackage.load(ruta)

print(f"guardado en: {ruta.name}")
print("reproduce los scores:",
      np.allclose(recargado.score(X_test.iloc[:500]), package.score(X_test.iloc[:500])))

## 5. Latencia: el número que un backtest nunca muestra

El Módulo 3 cronometró el scoring por lotes de 58.213 filas. Es la métrica correcta para un backtest y la equivocada para producción: ahí las transacciones llegan **de a una**, y el costo fijo por llamada domina sobre el costo marginal por fila.

Los dos números no se parecen.

In [ ]:
medir_latencia(recargado, X_test, n=200)

## 6. Cuántas alertas genera por día

El paquete promete una tasa de falsas alarmas. Con el volumen real del período, eso se traduce en una carga de trabajo concreta — que es lo que un equipo necesita saber antes de aceptar el sistema.

In [ ]:
carga = carga_diaria(recargado, X_test, steps, data["cutoff_step"])

print(f"mediana de alertas por día: {carga['alertas'].median():.0f}")
print(f"tasa media observada: {carga['tasa'].mean():.4%} (prometida: {recargado.alpha:.2%})")
carga.head(10)

## 7. Por qué se disparó esta alerta

Un analista que recibe una transacción marcada necesita saber qué la marcó. Sin eso la alerta no es accionable: no se puede confirmar, no se puede descartar rápido y no se puede explicar a un cliente que reclama.

De los dieciséis detectores, solo LODA trae atribución propia (Módulo 6). Esta implementación es **agnóstica al modelo**: no mira el interior del detector, solo lo consulta. Reemplaza una feature por su valor típico y mide cuánto cae el score.

In [ ]:
alertas = top_alerts(recargado, X_test, n=5)
alertas["es_fraude"] = np.asarray(y_test)[np.argsort(recargado.p_values(X_test))[:5]]
alertas

Dos límites del método que conviene tener presentes al leer esa columna:

- **Mide contribución marginal, no causalidad.** Si dos features están correlacionadas y juntas hacen anómala a la transacción, ocluir una sola puede no mover el score, y las dos aparecerían como irrelevantes. Es el mismo problema de los métodos de permutación.
- **La línea de base importa.** Reemplazar por la mediana pregunta *"¿qué pasa si esta transacción fuera típica en esta columna?"*, que es la pregunta correcta para explicar una alerta, pero no la única posible.

## 8. En qué se apoya el detector

Promediar la atribución sobre muchas transacciones da una lectura global: en qué se apoya el detector en general, no solo en un caso.

Sirve para detectar una fragilidad que ninguna métrica muestra — que toda la capacidad del detector descanse en una sola columna. Si esa columna cambia de definición aguas arriba, el sistema se degrada sin que nada avise.

In [ ]:
centro = recargado.scaler.transform(
    np.asarray(recargado.scaler.center_).reshape(1, -1)
).ravel()

global_importance(
    recargado.score_from_scaled,
    recargado.scaler.transform(X_test.iloc[:2000]),
    centro,
    recargado.feature_names,
)

## 9. Conclusiones

- **Un modelo serializado no es un sistema.** Las cinco piezas del paquete tienen que viajar juntas, y la que más silenciosamente rompe todo es el contrato de columnas.
- **El p-valor es lo que vuelve accionable al score.** Un número sin unidades no se puede comunicar a un analista ni comparar entre detectores; una probabilidad sí.
- **La latencia de producción no es la del backtest.** Medir el scoring por lotes y suponer que se traslada a transacciones sueltas es subestimar el costo real por un factor grande.
- **Elegir el detector por su puesto en el ranking es un error**, y este módulo lo hace concreto: se empaqueta el segundo por PR-AUC porque el primero no tiene un umbral utilizable.
- **Una alerta sin motivo no es accionable.** La oclusión cuesta una pasada de scoring por feature, así que se explica solo lo que se va a revisar — que es exactamente la cola acotada del Módulo 5.